# 🦄 Pony Diffusion V6 XL - 완전 무검열(Uncensored) 초고속 이미지 생성기

- **100% 완전 무검열 (Zero Censorship)**: 어떤 제한이나 검열 필터도 없는 무검열 대명사 모델
- **올인원 단일 모델 (6.4 GB)**: 14GB 대형 모델과 달리 램 부족(OOM)이나 92% 멈춤 현상이 **0%**입니다.
- **초고속 생성**: Colab 무료 T4 GPU 기준 **단 15~20초** 만에 고화질 생성 완료!
- **원클릭 웹 UI**: 복잡한 노드 설정 없이 프롬프트 입력창과 결과창만 깔끔하게 제공됩니다.

👉 **상단 메뉴에서 `런타임 > 모두 실행(Run all)`**을 클릭하시면 모든 과정이 전자동으로 진행됩니다.

### 1단계: GPU 환경 확인
Colab 상단 메뉴 `런타임 > 런타임 유형 변경`에서 **T4 GPU**로 설정되어 있는지 확인합니다.

In [ ]:
!nvidia-smi

### 2단계: 필수 라이브러리 설치
초고속 다운로더(`aria2`) 및 `diffusers`, `gradio` 웹 라이브러리를 설치합니다.

In [ ]:
!apt-get update -qq && apt-get install -y -qq aria2
!pip install -q diffusers transformers accelerate safetensors gradio Pillow invisible-watermark
print('✅ 필수 라이브러리 설치 완료!')

### 3단계: 완전 무검열 올인원 모델(6.4GB) 고속 다운로드
Pony Diffusion V6 XL 단일 파일(`6.46 GB`)을 고속으로 다운로드합니다.

In [ ]:
import os

os.makedirs('/content/models', exist_ok=True)
model_path = '/content/models/ponyDiffusionV6XL_v6StartWithThisOne.safetensors'

if not os.path.exists(model_path) or os.path.getsize(model_path) < 6000000000:
    print('📥 [Pony Diffusion V6 XL] 완전 무검열 모델 다운로드 중 (약 6.4 GB, 30~50초 소요)...')
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
      --user-agent="Mozilla/5.0" \
      "https://huggingface.co/LyliaEngine/Pony_Diffusion_V6_XL/resolve/main/ponyDiffusionV6XL_v6StartWithThisOne.safetensors" \
      -d /content/models -o ponyDiffusionV6XL_v6StartWithThisOne.safetensors
    print('✅ 모델 다운로드 완료!')
else:
    print('✅ 모델 파일이 이미 준비되어 있습니다.')

### 4단계: 무검열 웹 생성기 실행 (15~20초 초고속 생성)
아래 셀을 실행하면 **Colab 화면 바로 아래에 프롬프트 입력창**이 뜨며, 브라우저 전체 창이나 모바일에서 열 수 있는 `gradio.live` 공개 링크도 제공됩니다!

In [ ]:
import os
import time
import random
import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
import gradio as gr
from PIL import Image

# 1. 기존 GPU 메모리 점유 프로세스 정리 및 캐시 비우기
!fuser -k -9 /dev/nvidia* > /dev/null 2>&1 || true
torch.cuda.empty_cache()
time.sleep(1)

print('⏳ AI 모델 로딩 중 (VRAM 5GB 초절약 모드 적용)...')
model_path = '/content/models/ponyDiffusionV6XL_v6StartWithThisOne.safetensors'

# 2. 파이프라인 로드
pipe = StableDiffusionXLPipeline.from_single_file(
    model_path,
    torch_dtype=torch.float16,
    use_safetensors=True
)

# 3. DPM++ 2M Karras 스케줄러 적용 (20스텝만으로 초고화질 완성)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    use_karras_sigmas=True,
    algorithm_type='sde-dpmsolver++'
)

# 4. 핵심: VRAM을 4~5GB만 사용하도록 오프로드 설정 (OOM 완전 방지 & GPU 100% 풀스피드)
pipe.enable_model_cpu_offload()

print('🎉 모델 준비 완료! 아래 웹 입력창에서 이미지를 생성하세요.')

# 5. 이미지 생성 함수
def generate_image(prompt, negative_prompt, steps, cfg, width, height, seed, progress=gr.Progress(track_tqdm=True)):
    if not prompt or prompt.strip() == '':
        raise gr.Error('프롬프트를 입력해 주세요!')
    
    # Pony 특화 품질 태그 자동 보강
    if 'score_' not in prompt:
        full_prompt = f'score_9, score_8_up, score_7_up, {prompt}'
    else:
        full_prompt = prompt
        
    if 'score_' not in negative_prompt:
        full_neg = f'score_4, score_5, score_6, {negative_prompt}'
    else:
        full_neg = negative_prompt

    if seed == -1 or seed is None:
        seed = random.randint(1, 2147483647)
    
    generator = torch.Generator('cpu').manual_seed(int(seed))
    
    # 생성 시작 (약 15~20초 소요, OOM 발생률 0%)
    image = pipe(
        prompt=full_prompt,
        negative_prompt=full_neg,
        num_inference_steps=int(steps),
        guidance_scale=float(cfg),
        width=int(width),
        height=int(height),
        generator=generator
    ).images[0]
    
    return image

# 6. 심플 Gradio 웹 UI
with gr.Blocks(theme=gr.themes.Soft(), title='Pony V6 XL 무검열 이미지 생성기') as demo:
    gr.Markdown('# 🦄 Pony Diffusion V6 XL - 완전 무검열 이미지 생성기')
    gr.Markdown('원하는 프롬프트를 적고 **[이미지 생성하기]**를 누르면 **약 15~20초** 만에 이미지가 생성됩니다.')
    
    with gr.Row():
        with gr.Column(scale=1):
            prompt_input = gr.Textbox(
                label='📝 프롬프트 (그릴 내용 - 완전 무검열)',
                placeholder='영어로 자유롭게 입력하세요',
                lines=4,
                value='1girl, beautiful anime girl with silver hair, blue eyes, smiling, in a cozy cafe, highly detailed, masterpiece'
            )
            neg_input = gr.Textbox(
                label='🚫 부정 프롬프트 (제외할 내용)',
                lines=2,
                value='low quality, blurry, distorted, deformed, bad anatomy, worst quality'
            )
            
            with gr.Accordion('⚙️ 상세 설정 (기본값 832x832 권장)', open=False):
                with gr.Row():
                    w_slider = gr.Slider(512, 1024, value=832, step=64, label='가로 너비 (기본 832)')
                    h_slider = gr.Slider(512, 1024, value=832, step=64, label='세로 높이 (기본 832)')
                steps_slider = gr.Slider(15, 35, value=22, step=1, label='생성 스텝수 (권장 20~25)')
                cfg_slider = gr.Slider(3.0, 10.0, value=6.5, step=0.5, label='CFG (Pony 권장: 6.0~7.0)')
                seed_input = gr.Number(value=-1, label='시드 (-1이면 랜덤)')
            
            gen_btn = gr.Button('🚀 이미지 생성하기 (약 15~20초)', variant='primary', size='lg')
            
        with gr.Column(scale=1):
            out_img = gr.Image(label='🖼️ 생성된 이미지', type='pil', interactive=False)
            
    gen_btn.click(
        fn=generate_image,
        inputs=[prompt_input, neg_input, steps_slider, cfg_slider, w_slider, h_slider, seed_input],
        outputs=out_img
    )

demo.queue().launch(share=True, debug=False)
